Resultaten wegschrijven in genormaliseerd datamodel. Voorlopig gesimuleerd als sqlite. Connectie met LSVI databank of andere masterdata nog uit te klaren?

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sqlite3
from datetime import datetime
import pandas as pd

from arcgis.gis import GIS

import os
import sys

# Get the absolute path of the folder above the notebook
parent_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))

if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

from src import utils

In [74]:
pd.set_option('display.max_columns', None)  


### Connectie naar databank

In [3]:
# Aanmaken 
def init_database(db_path="lsvi_resultaten_slank.sqlite"):
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    cursor.execute("PRAGMA foreign_keys = ON;")
    
    # Tabel 1: WaarnemingEvent
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS waarneming_event (
        collectie_id TEXT PRIMARY KEY,
        global_id TEXT,
        user_name TEXT,
        bwk_plot_id TEXT,
        bwk_globalid TEXT,
        bwk_centroid_x REAL,
        bwk_centroid_y REAL,
        locatie_x REAL,
        locatie_y REAL,
        EPSG INTEGER,
        doel_habitattype TEXT,
        created_date DATETIME,
        last_edited_date DATETIME,
        timestamp_measurement DATETIME
    );
    """)
    
    # Tabel 2: Resultaat (Nu ook voor de losse soorten!)
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS resultaat (
        resultaat_id INTEGER PRIMARY KEY AUTOINCREMENT,
        collectie_id TEXT,
        voorwaarde_id INTEGER,
        vraag_id TEXT,
        subvraag TEXT,
        waarde_tekst TEXT,
        waarde_numeriek REAL,
        FOREIGN KEY (collectie_id) REFERENCES waarneming_event(collectie_id) ON DELETE CASCADE
    );
    """)
    conn.commit()
    return conn

# Helperfunctie om ArcGIS Epoch Milliseconds om te zetten naar een ISO string
def format_agol_date(epoch_ms):
    if epoch_ms is None:
        return None
    # AGOL timestamps zijn in milliseconden, Python verwacht seconden
    return datetime.fromtimestamp(epoch_ms / 1000.0).strftime('%Y-%m-%d %H:%M:%S')



### Run ETL

In [55]:
# Uitvoering
# Joost to do: credentials uit key vault of environment
# Load secrets into environment
LOCAL_DB = "G:\\Mijn Drive\\keepass_db.kdbx"
ENTRY_TITLE = "AGOL"
AGOL_URL = "https://gisservices.inbo.be/portal"
FEATURE_LAYER_NAME = "xlsform_hab1-4"

AGOL_USER, AGOL_PASS = utils.load_keepass_credentials(
        db_path=LOCAL_DB, entry_name=ENTRY_TITLE
    )

today = datetime.now().strftime("%Y%m%d")
db_path = f"../output/lsvi_resultaten_{today}.sqlite"

# Verbinden met AGOL
print(f"Verbinden met {AGOL_URL}...")
gis = GIS(AGOL_URL, AGOL_USER, AGOL_PASS)

# Not used because simplified export to CSV
# conn = init_database(db_path)
# cursor = conn.cursor()

feature_layer_list = ['xlsform_hab1-4'] #, 'xlsform_hab5-7', 'xlsform_hab9']

ℹ️ Inloggegevens voor 'AGOL' zijn al actief in deze sessie. Prompt overgeslagen.
Verbinden met https://gisservices.inbo.be/portal...


In [76]:
target_bwk_globalid = '{A1B16158-BE17-4857-BB07-EB3FCD3C6050}'

filtered_features = [
    f for f in features_result.features
    if (f.attributes or {}).get("bwk_globalid") == target_bwk_globalid
]

print(f"Aantal matches: {len(filtered_features)}")
filtered_features[0].attributes if filtered_features else None

Aantal matches: 1


{'vrg_370_2190_mp_matrix_685': None,
 'vrg_368_1330_hpr_matrix_4240': None,
 'vrg_579_1330_da_matrix_4207': None,
 'vrg_434_4030_matrix_4167': None,
 'vrg_608_4010_matrix_4296': None,
 'vrg_615_2130_had_matrix_501': None,
 'vrg_1016_3130_na': None,
 'vrg_601_2160_matrix_1925': None,
 'vrg_711_1320': None,
 'vrg_599_1310_zv_matrix_2158': None,
 'vrg_2585_2190_a': None,
 'vrg_1725_2130_had': None,
 'vrg_1492_2130_hd': None,
 'niet_grondig_2180': None,
 'vrg_466_2150_matrix_1643': None,
 'vrg_1554_2130_hd': None,
 'vrg_1317_1310_zk': None,
 'niet_grondig_3150': None,
 'niet_grondig_3270': None,
 'habitat_keuze': '4010',
 'vrg_466_2150_matrix_2715': None,
 'vrg_370_2190_mp_matrix_692': None,
 'vrg_608_4010_matrix_1902': None,
 'vrg_613_2120_matrix_729': None,
 'vrg_596_2130_had_matrix_5': None,
 'vrg_997_2170': None,
 'vrg_1226_2190_a': None,
 'vrg_2057_3160': None,
 'vrg_471_3110_matrix_2089': None,
 'vrg_350_3270_matrix_2606': None,
 'vrg_348_2310_matrix_360': None,
 'vrg_694_2120_matrix

In [64]:
type(geom)

dict

In [78]:
# Loop over feature layers
for FEATURE_LAYER_NAME in feature_layer_list:
    # Get feature layer id from name
    layer_items = gis.content.search(FEATURE_LAYER_NAME, item_type='Feature Layer', max_items=10)
    # Exclude item in list if ends with _form
    layer_items = [item for item in layer_items if not item.title.endswith('_form')]
    feature_layer_item_id = layer_items[0].id if layer_items else None
    print(f"Feature layer {feature_layer_item_id} downloaden...")
    feature_layer = layer_items[0].layers[0]

    # Vraag alle records op (1=1) inclusief WGS84 geometrie (out_sr=4326)
    features_result = feature_layer.query(where="1=1", out_sr=4326)
    print(f"Succesvol {len(features_result.features)} features gedownload. Start verwerking...")

    # Field aliases contain more information about question and their labels
    field_aliases = {
        f.name: f.alias for f in feature_layer.properties.fields
    } if hasattr(feature_layer, 'properties') else {}

    # Initialize rows list for output
    rows = []

    # Loop over each feature in feature layer
    for feature in features_result.features:
        geom = feature.geometry if feature.geometry else {}
        attrs = feature.attributes if feature.attributes else {}
        
        # Controleer of de cruciale collectie_id aanwezig is
        collectie_id = attrs.get('collectie_id')
        if not collectie_id:
            collectie_id = attrs.get('globalid')
            
        # Metadata extraheren en parsen
        global_id = attrs.get('globalid')
        created_user = attrs.get('created_user')
        last_edited_user = attrs.get('last_edited_user')
        habitat_keuze = attrs.get('habitat_keuze')
        bwk_plot_id = attrs.get('bwk_plot_id')
        bwk_globalid = attrs.get('bwk_globalid')
        bwk_centroid_x = attrs.get('bwk_centroid_x')
        bwk_centroid_y = attrs.get('bwk_centroid_y')
        
        created_date_txt = format_agol_date(attrs.get('datum'))
        last_edited_date_txt = format_agol_date(attrs.get('last_edited_date'))
        
        # Tijdstip van het bezoek bepalen (Datum-veld + Uur-veld combineren)
        datum_txt = format_agol_date(attrs.get('datum'))
        uur_txt = attrs.get('uur')  # string zoals "14:59"
        tijdstip_waarneming = f"{datum_txt.split(' ')[0]} {uur_txt}" if datum_txt and uur_txt else datum_txt

        # Opmerkingen
        opmerkingen = attrs.get('opmerkingen')

        # Niet grondig doorzocht
        niet_grondig_doorzocht = attrs.get(f"niet_grondig_{habitat_keuze}")

        # Geometrie
        x = geom.get('x')
        y = geom.get('y')
        epsg = geom.get('spatialReference', {}).get('wkid')

        metadata = {
                    'collectie_id': collectie_id,
                    'global_id': global_id,
                    'created_user': created_user,
                    'last_edited_user': last_edited_user,
                    'bwk_plot_id': bwk_plot_id,
                    'bwk_globalid': bwk_globalid,
                    'bwk_centroid_x_l72': bwk_centroid_x,
                    'bwk_centroid_y_l72': bwk_centroid_y,
                    'locatie_x_wgs84': x,
                    'locatie_y_wgs84': y,
                    'epsg': epsg,
                    'doel_habitattype': habitat_keuze,
                    'created_date': created_date_txt,
                    'last_edited_date': last_edited_date_txt,
                    'timestamp_measurement': tijdstip_waarneming,
                    'opmerkingen': opmerkingen,
                    'niet_grondig_doorzocht': niet_grondig_doorzocht
                }
        
        # Loop dynamisch door alle kolommen van de feature
        for key, value in attrs.items():
            # Sla lege cellen, systeemenmerken en layout-hulpmiddelen over
            if value is None:
                continue

            # Extraheer het getal (VoorwaardeID) als de kolom begint met 'vrg_'
            voorwaarde_id = None
            habitattype = None
            subvraag = None

            if key.startswith("vrg_"):
                clean_key = key[4:]

                # Controleer of het een matrixvraag betreft
                if "_matrix_" in clean_key:
                    # Bv. '736_1310_zv_matrix_0' -> base_part = '736_1310_zv'
                    base_part, groep_id = clean_key.split("_matrix_") # if groep = Sleutelsoorten --> groep_id is taxon_id
                    # Ophalen van de weergavenaam (subvraag) uit onze lookup dict
                    subvraag = field_aliases.get(key)
                else:
                    base_part = clean_key
                    groep_id = None
                    subvraag = None  # Geen matrix, dus geen subvraag

                # Ontleed voorwaarde_id en habitattype uit base_part (bv. '712_1310_zk')
                parts = base_part.split("_", 1)
                voorwaarde_id = int(parts[0]) if parts[0].isdigit() else None
                habitattype = parts[1] if len(parts) > 1 else None
            
                # Gegevenstype bepalen voor waarde_numeriek
                waarde_numeriek = None
                if isinstance(value, (int, float)):
                    waarde_numeriek = float(value)
                elif isinstance(value, str):
                    try:
                        waarde_numeriek = float(value)
                    except ValueError:
                        pass

                # Afhandeling van select_multiple (komma-gescheiden waarden)
                if isinstance(value, str) and "," in value:
                    # Comma-separated → per soort 1 rij
                    soorten = [s.strip() for s in value.split(",")]
                    for soort in soorten:
                        if soort:
                            row = {
                                **metadata,  # Unpack metadata
                                'voorwaarde_id': voorwaarde_id,
                                'habitattype': habitattype,
                                'groep_id': groep_id,
                                'subvraag': subvraag,
                                'vraag_id': key,
                                'waarde_tekst': soort,
                                'waarde_numeriek': None
                            }
                            rows.append(row)
                else:
                    # Enkele waarde (numeriek of tekst)
                    row = {
                        **metadata,  # Unpack metadata
                        'voorwaarde_id': voorwaarde_id,
                        'habitattype': habitattype,
                        'groep_id': groep_id,
                        'subvraag': subvraag,
                        'vraag_id': key,
                        'waarde_tekst': str(value) if not waarde_numeriek else None,
                        'waarde_numeriek': waarde_numeriek
                    }
                    rows.append(row)
                    
# Eind van loop
df_resultaat = pd.DataFrame(rows)  
print("Export afgerond. De resultaten zijn opgeslagen in een DataFrame.")

Feature layer fadd6d9740ca4dd0bac2bf937038775c downloaden...
Succesvol 9 features gedownload. Start verwerking...
Export afgerond. De resultaten zijn opgeslagen in een DataFrame.


In [81]:

df_resultaat.to_csv(f"../output/lsvi_resultaten_{today}.csv", index=False, encoding='utf-8-sig')

In [79]:
df_resultaat[df_resultaat.bwk_globalid == '{A1B16158-BE17-4857-BB07-EB3FCD3C6050}']
# df_resultaat[df_resultaat.bwk_globalid == '{21760360-76C6-4830-AA1F-E062663F429E}']

,collectie_id,global_id,created_user,last_edited_user,bwk_plot_id,bwk_globalid,bwk_centroid_x_l72,bwk_centroid_y_l72,locatie_x_wgs84,locatie_y_wgs84,epsg,doel_habitattype,created_date,last_edited_date,timestamp_measurement,opmerkingen,niet_grondig_doorzocht,voorwaarde_id,habitattype,groep_id,subvraag,vraag_id,waarde_tekst,waarde_numeriek
18,{3FAE5F0D-3837-4309-A65D-14DC58569290},{3FAE5F0D-3837-4309-A65D-14DC58569290},joost.neujens@inbo.be,joost.neujens@inbo.be,NaN,{A1B16158-BE17-4857-BB07-EB3FCD3C6050},4937909396439,669325335965363,0.0,0.0,4326,4010,2026-07-28 12:00:00,2026-07-28 15:27:47,2026-07-28 15:27,Test,ja,608,4010,468,Bruine snavelbies,vrg_608_4010_matrix_468,wt,NaN
19,{3FAE5F0D-3837-4309-A65D-14DC58569290},{3FAE5F0D-3837-4309-A65D-14DC58569290},joost.neujens@inbo.be,joost.neujens@inbo.be,NaN,{A1B16158-BE17-4857-BB07-EB3FCD3C6050},4937909396439,669325335965363,0.0,0.0,4326,4010,2026-07-28 12:00:00,2026-07-28 15:27:47,2026-07-28 15:27,Test,ja,634,4010,2702,Wrattig veenmos,vrg_634_4010_matrix_2702,zs,NaN
20,{3FAE5F0D-3837-4309-A65D-14DC58569290},{3FAE5F0D-3837-4309-A65D-14DC58569290},joost.neujens@inbo.be,joost.neujens@inbo.be,NaN,{A1B16158-BE17-4857-BB07-EB3FCD3C6050},4937909396439,669325335965363,0.0,0.0,4326,4010,2026-07-28 12:00:00,2026-07-28 15:27:47,2026-07-28 15:27,Test,ja,634,4010,2708,Week veenmos,vrg_634_4010_matrix_2708,zs,NaN
21,{3FAE5F0D-3837-4309-A65D-14DC58569290},{3FAE5F0D-3837-4309-A65D-14DC58569290},joost.neujens@inbo.be,joost.neujens@inbo.be,NaN,{A1B16158-BE17-4857-BB07-EB3FCD3C6050},4937909396439,669325335965363,0.0,0.0,4326,4010,2026-07-28 12:00:00,2026-07-28 15:27:47,2026-07-28 15:27,Test,ja,780,4010,NaN,NaN,vrg_780_4010,60_70perc,NaN
22,{3FAE5F0D-3837-4309-A65D-14DC58569290},{3FAE5F0D-3837-4309-A65D-14DC58569290},joost.neujens@inbo.be,joost.neujens@inbo.be,NaN,{A1B16158-BE17-4857-BB07-EB3FCD3C6050},4937909396439,669325335965363,0.0,0.0,4326,4010,2026-07-28 12:00:00,2026-07-28 15:27:47,2026-07-28 15:27,Test,ja,1583,4010,NaN,NaN,vrg_1583_4010,afw,NaN
23,{3FAE5F0D-3837-4309-A65D-14DC58569290},{3FAE5F0D-3837-4309-A65D-14DC58569290},joost.neujens@inbo.be,joost.neujens@inbo.be,NaN,{A1B16158-BE17-4857-BB07-EB3FCD3C6050},4937909396439,669325335965363,0.0,0.0,4326,4010,2026-07-28 12:00:00,2026-07-28 15:27:47,2026-07-28 15:27,Test,ja,608,4010,1249,Tweenervige zegge,vrg_608_4010_matrix_1249,wt,NaN
24,{3FAE5F0D-3837-4309-A65D-14DC58569290},{3FAE5F0D-3837-4309-A65D-14DC58569290},joost.neujens@inbo.be,joost.neujens@inbo.be,NaN,{A1B16158-BE17-4857-BB07-EB3FCD3C6050},4937909396439,669325335965363,0.0,0.0,4326,4010,2026-07-28 12:00:00,2026-07-28 15:27:47,2026-07-28 15:27,Test,ja,634,4010,4008,Glanzend veenmos,vrg_634_4010_matrix_4008,s,NaN
25,{3FAE5F0D-3837-4309-A65D-14DC58569290},{3FAE5F0D-3837-4309-A65D-14DC58569290},joost.neujens@inbo.be,joost.neujens@inbo.be,NaN,{A1B16158-BE17-4857-BB07-EB3FCD3C6050},4937909396439,669325335965363,0.0,0.0,4326,4010,2026-07-28 12:00:00,2026-07-28 15:27:47,2026-07-28 15:27,Test,ja,1747,4010,NaN,NaN,vrg_1747_4010,afw,NaN


- 1 collectie_id is 1 survey. 
- Deze collectie is gelink aan een plot_ID (bwk laag) indien ingevuld, maar ook via de globalid van BWK en centroide van polygoon
- We houden zowel deze centroide bij (L72) als de locatie van het device bij openen van de survey (WGS84).
    De centroide coordinaten hebben een decimaal punt ('.') hetgeen automatisch verwijderd wordt bij opslag in databank wegens regio/taal settings. Workaround moet nog getest worden bij nieuwe versie van de survey.
- De voorwaarde id komt overeen met voorwaarde id uit invoervereisten
- Indien 1 voorwaarde (vraag), 1 antwoord kent, zal je het antwoord in kolom waarde_tekst of waarde_numeriek terugvinden.
- In geval van select multiple, gaan de meerdere antwoorden gesplitst worden over meerdere rijen (met dezelfde collectieID en voorwaarde ID)
- In geval van matrixvraag: voor eenzelfde voorwaarde id ga je meerdere rijen terugvinden met een groep_id ingevuld en bijhorende subvraag. Indien de matrixvraag voor sleutelsoorten is, zal de groep_id de taxon_id zijn uit gekoppelde soortenlijst en subvraag de nederlandstalige naam van deze soort. Indien de matrixvraag een andere groepering kent, zal de groep_id verwijzen naar de gegeven groep in de invoervereisten (bv. groeiklasse 1 in groep van 'groeiklassen bomen' kent id = 21).
- Veldmedewerker kan 'ja' of 'nee' aanduiden bij de vraag 'Niet grondig onderzocht'. ja --> snel ingevuld maar niet volledig onderzocht. Een extra veld met opmerkingen is ook mogelijk.

### Old code using sqlite export

In [ ]:
# # Uitvoering
# # Joost to do: credentials uit key vault of environment
# # Load secrets into environment
# LOCAL_DB = "G:\\Mijn Drive\\keepass_db.kdbx"
# ENTRY_TITLE = "AGOL"
# AGOL_URL = "https://gisservices.inbo.be/portal"
# FEATURE_LAYER_NAME = "xlsform_hab1-4"

# AGOL_USER, AGOL_PASS = utils.load_keepass_credentials(
#         db_path=LOCAL_DB, entry_name=ENTRY_TITLE
#     )

# today = datetime.now().strftime("%Y%m%d")
# db_path = f"../output/lsvi_resultaten_{today}.sqlite"

# # Verbinden met AGOL
# print(f"Verbinden met {AGOL_URL}...")
# gis = GIS(AGOL_URL, AGOL_USER, AGOL_PASS)

# # Not used because simplified export to CSV
# # conn = init_database(db_path)
# # cursor = conn.cursor()

# feature_layer_list = ['xlsform_hab1-4', 'xlsform_hab5-7', 'xlsform_hab9']

# for FEATURE_LAYER_NAME in feature_layer_list:
#     # Get feature layer id from name
#     layer_items = gis.content.search(FEATURE_LAYER_NAME, item_type='Feature Layer', max_items=10)
#     # Exclude item in list if ends with _form
#     layer_items = [item for item in layer_items if not item.title.endswith('_form')]
#     feature_layer_item_id = layer_items[0].id if layer_items else None
#     print(f"Feature layer {feature_layer_item_id} downloaden...")
#     feature_layer = layer_items[0].layers[0]

#     # Vraag alle records op (1=1) inclusief WGS84 geometrie (out_sr=4326)
#     features_result = feature_layer.query(where="1=1", out_sr=4326)
#     print(f"Succesvol {len(features_result.features)} features gedownload. Start verwerking...")

#     field_aliases = {
#         f.name: f.alias for f in feature_layer.properties.fields
#     } if hasattr(feature_layer, 'properties') else {}

#     # Loop over each feature in feature layer
#     for feature in features_result.features:
#         geom = feature.geometry if feature.geometry else {}
#         attrs = feature.attributes if feature.attributes else {}
        
#         # Controleer of de cruciale collectie_id aanwezig is
#         collectie_id = attrs.get('collectie_id')
#         if not collectie_id:
#             collectie_id = attrs.get('globalid')
            
#         # Metadata extraheren en parsen
#         global_id = attrs.get('globalid')
#         user_name = attrs.get('username')
#         habitat_keuze = attrs.get('habitat_keuze')
#         bwk_plot_id = attrs.get('bwk_plot_id')
#         bwk_globalid = attrs.get('bwk_globalid')
#         bwk_centroid_x = attrs.get('bwk_centroid_x')
#         bwk_centroid_y = attrs.get('bwk_centroid_y')
        
#         created_date_txt = format_agol_date(attrs.get('datum'))
#         last_edited_date_txt = format_agol_date(attrs.get('last_edited_date'))
        
#         # Tijdstip van het bezoek bepalen (Datum-veld + Uur-veld combineren)
#         datum_txt = format_agol_date(attrs.get('datum'))
#         uur_txt = attrs.get('uur')  # string zoals "14:59"
#         tijdstip_waarneming = f"{datum_txt.split(' ')[0]} {uur_txt}" if datum_txt and uur_txt else datum_txt

#         # Geometrie
#         x = geom.get('x')
#         y = geom.get('y')
#         epsg = geom.get('spatialReference', {}).get('wkid')

#         # WaarnemingEvent wegschrijven
#         cursor.execute("""
#             INSERT INTO waarneming_event (
#                 collectie_id, 
#                 global_id, 
#                 user_name, 
#                 bwk_plot_id,
#                 bwk_globalid,
#                 bwk_centroid_x,
#                 bwk_centroid_y,
#                 locatie_x, 
#                 locatie_y, 
#                 epsg, 
#                 doel_habitattype,  
#                 created_date, 
#                 last_edited_date,
#                 timestamp_measurement
#             ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
#             ON CONFLICT(collectie_id) DO UPDATE SET
#                 global_id=excluded.global_id,
#                 user_name=excluded.user_name,
#                 bwk_plot_id=excluded.bwk_plot_id,
#                 bwk_globalid=excluded.bwk_globalid,
#                 bwk_centroid_x=excluded.bwk_centroid_x,
#                 bwk_centroid_y=excluded.bwk_centroid_y,
#                 locatie_x=excluded.locatie_x,
#                 locatie_y=excluded.locatie_y,
#                 epsg=excluded.epsg,
#                 doel_habitattype=excluded.doel_habitattype,
#                 created_date=excluded.created_date,
#                 last_edited_date=excluded.last_edited_date,
#                 timestamp_measurement=excluded.timestamp_measurement;
#             """, (collectie_id, global_id, user_name, bwk_plot_id, bwk_globalid, bwk_centroid_x, bwk_centroid_y, x, y, epsg, habitat_keuze, created_date_txt, last_edited_date_txt, tijdstip_waarneming))
        
#         # Resultaten wegschrijven
#         # Eerst collectieID wissen voor overschrijven
#         cursor.execute("DELETE FROM resultaat WHERE collectie_id = ?", (collectie_id,))

#         # Loop dynamisch door alle kolommen van de feature
#         for key, value in attrs.items():
#             # Sla lege cellen, systeemenmerken en layout-hulpmiddelen over
#             if value is None:
#                 continue

#             # Extraheer het getal (VoorwaardeID) als de kolom begint met 'vrg_'
#             voorwaarde_id = None
#             habitattype = None
#             subvraag = None

#             if key.startswith("vrg_"):
#                 clean_key = key[4:]

#                 # Controleer of het een matrixvraag betreft
#                 if "_matrix_" in clean_key:
#                     # Bv. '736_1310_zv_matrix_0' -> base_part = '736_1310_zv'
#                     base_part = clean_key.split("_matrix_")[0]
#                     # Ophalen van de weergavenaam (subvraag) uit onze lookup dict
#                     subvraag = field_aliases.get(key)
#                 else:
#                     base_part = clean_key
#                     subvraag = None  # Geen matrix, dus geen subvraag

#                 # Ontleed voorwaarde_id en habitattype uit base_part (bv. '712_1310_zk')
#                 parts = base_part.split("_", 1)
#                 if parts[0].isdigit():
#                     voorwaarde_id = int(parts[0])
#                     if len(parts) > 1:
#                         habitattype = parts[1]  # Bv. '1310_zk' of '1310_zv'
            
#                 # Gegevenstype bepalen voor waarde_numeriek
#                 waarde_numeriek = None
#                 if isinstance(value, (int, float)):
#                     waarde_numeriek = float(value)
#                 elif isinstance(value, str):
#                     try:
#                         waarde_numeriek = float(value)
#                     except ValueError:
#                         pass

#                 # Afhandeling van select_multiple (komma-gescheiden waarden)
#                 if isinstance(value, str) and "," in value:
#                     soorten = [s.strip() for s in value.split(",")]
#                     for soort in soorten:
#                         if soort:
#                             cursor.execute("""
#                                 INSERT INTO resultaat (
#                                     collectie_id, voorwaarde_id, vraag_id, subvraag, waarde_tekst, waarde_numeriek
#                                 )
#                                 VALUES (?, ?, ?, ?, ?, ?);
#                             """, (collectie_id, voorwaarde_id, key, subvraag,  soort, waarde_numeriek))
#                 else:
#                     # Normale vraag / enkele waarde
#                     cursor.execute("""
#                         INSERT INTO resultaat (
#                             collectie_id, voorwaarde_id, vraag_id, subvraag, waarde_tekst, waarde_numeriek
#                         )
#                         VALUES (?, ?, ?, ?, ?, ?);
#                     """, (collectie_id, voorwaarde_id, key, subvraag, str(value), waarde_numeriek))
    
# print("ETL afgerond. De SQLite database is volledig up-to-date.")
# conn.commit()
# conn.close()
    